# Yield Phase Diagrams

Loads minimised n-ring structures (from `ring_minimization.ipynb`) and yield data
(from `run_yield_simulation_cluster.py` outputs) to produce:

- **Panel A** — per-structure yield heatmaps with validity contours
- **Panel B** — relative yield (triangle vs. square)

Parameter grid aligned with cluster config: `YIELD_ALPHAS_DEG` × `YIELD_ZETAS`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import warnings

from base.grids import YIELD_ALPHAS_DEG, YIELD_ZETAS

## Parameter grid

In [ ]:
# Grid aligned with cluster simulation
opening_angles_deg = YIELD_ALPHAS_DEG          # shape (10,)
zetas              = YIELD_ZETAS               # shape (10,), log-spaced
E_AB_values        = 2.0 * zetas               # E_AB used in ring_minimization

N_ANGLE = len(opening_angles_deg)
N_ZETA  = len(zetas)

STRUCTURES = ['triangle', 'square', 'pentagon', 'hexagon', 'heptagon', 'octagon']
N_RINGS    = [3, 4, 5, 6, 7, 8]

YIELD_DIR = Path('yield_results_cluster')

print(f"Opening angles (deg): {np.round(opening_angles_deg, 1)}")
print(f"Zetas:                {np.round(zetas, 3)}")
print(f"Grid: {N_ANGLE} x {N_ZETA} = {N_ANGLE*N_ZETA} jobs")

## Load yield data from cluster output files

In [ ]:
# yields[structure] has shape (N_ANGLE, N_ZETA); NaN where file is missing.
yields = {s: np.full((N_ANGLE, N_ZETA), np.nan) for s in STRUCTURES}
missing = []

for alpha_idx, alpha in enumerate(opening_angles_deg):
    for zeta_idx, zeta in enumerate(zetas):
        job_id = alpha_idx * N_ZETA + zeta_idx
        fname = YIELD_DIR / f'job_{job_id:04d}_alpha{alpha:.0f}_zeta{zeta:.2f}_yields.npz'
        if not fname.exists():
            missing.append(str(fname))
            continue
        data = np.load(fname)
        for s in STRUCTURES:
            key = f'avg_Y_{s}'
            if key in data:
                yields[s][alpha_idx, zeta_idx] = float(data[key])

if missing:
    warnings.warn(f"{len(missing)} yield file(s) missing (first: {missing[0]})")
else:
    print("All yield files loaded successfully.")

# Quick summary
for s in STRUCTURES:
    valid = np.sum(~np.isnan(yields[s]))
    print(f"  {s:10s}: {valid}/{N_ANGLE*N_ZETA} grid points loaded")

## Load ring validity contours from ring_minimization.ipynb output

In [ ]:
# validity[n] has shape (N_ANGLE, N_ZETA): 1 = structurally valid ring, 0 = not.
# E_AB_values = 2*zetas, so the grid aligns exactly.
validity = {}

for n in N_RINGS:
    fname = f'minimized_n{n}_ring.npz'
    if os.path.exists(fname):
        d = np.load(fname)
        validity[n] = d['validity']   # (N_ANGLE, N_EAB) = (N_ANGLE, N_ZETA)
        print(f"  n={n}: loaded, valid fraction = {validity[n].mean():.2f}")
    else:
        validity[n] = None
        print(f"  n={n}: minimized_n{n}_ring.npz not found — no validity contour")

## Panel A — Per-structure yield heatmaps with validity contours

In [ ]:
structure_labels = {
    'triangle': 'Triangle (n=3)',
    'square':   'Square (n=4)',
    'pentagon': 'Pentagon (n=5)',
    'hexagon':  'Hexagon (n=6)',
    'heptagon': 'Heptagon (n=7)',
    'octagon':  'Octagon (n=8)',
}
n_for_structure = dict(zip(STRUCTURES, N_RINGS))

# Axes: opening angle on x, zeta (log-scale) on y.
# imshow with origin='lower' → rows = zeta (ascending), cols = angle (ascending).
# extent uses bin edges for pixel-accurate alignment.
angle_edges = np.linspace(opening_angles_deg[0], opening_angles_deg[-1], N_ANGLE + 1)
zeta_log_edges = np.linspace(np.log10(zetas[0]), np.log10(zetas[-1]), N_ZETA + 1)

# Mesh centres for validity contour overlay
angle_mesh, zeta_mesh = np.meshgrid(opening_angles_deg, np.log10(zetas))

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
fig.suptitle('Panel A: Per-structure yield', fontsize=15)

for ax, s in zip(axes.flat, STRUCTURES):
    n = n_for_structure[s]
    Y = yields[s].T          # (N_ZETA, N_ANGLE) for imshow
    im = ax.imshow(
        Y, origin='lower', aspect='auto',
        extent=[angle_edges[0], angle_edges[-1], zeta_log_edges[0], zeta_log_edges[-1]],
        cmap='plasma', vmin=0, vmax=1,
        interpolation='nearest',
    )
    plt.colorbar(im, ax=ax, label='Yield')

    # Validity contour (0.5 level separates valid/invalid)
    if validity.get(n) is not None:
        V = validity[n].T   # (N_ZETA, N_ANGLE)
        ax.contour(angle_mesh, zeta_mesh, V, levels=[0.5],
                   colors='cyan', linewidths=1.5, linestyles='--')

    # Log-scale y-tick labels
    ytick_log = np.linspace(zeta_log_edges[0], zeta_log_edges[-1], 5)
    ax.set_yticks(ytick_log)
    ax.set_yticklabels([f'{10**v:.2g}' for v in ytick_log])

    ax.set_xlabel('Opening angle (deg)')
    ax.set_ylabel('ζ  (selectivity)')
    ax.set_title(structure_labels[s])

plt.tight_layout()
plt.savefig('yield_panel_A.png', dpi=150, bbox_inches='tight')
plt.show()

## Panel B — Relative yield: (Y_tri − Y_sq) / (Y_tri + Y_sq)

In [ ]:
Y_tri = yields['triangle']   # (N_ANGLE, N_ZETA)
Y_sq  = yields['square']

denom = Y_tri + Y_sq
with np.errstate(invalid='ignore', divide='ignore'):
    rel_yield = np.where(denom > 0, (Y_tri - Y_sq) / denom, np.nan)

fig, ax = plt.subplots(figsize=(7, 5))
fig.suptitle('Panel B: Relative yield  (Y_tri − Y_sq) / (Y_tri + Y_sq)', fontsize=13)

im = ax.imshow(
    rel_yield.T, origin='lower', aspect='auto',
    extent=[angle_edges[0], angle_edges[-1], zeta_log_edges[0], zeta_log_edges[-1]],
    cmap='RdBu', vmin=-1, vmax=1,
    interpolation='nearest',
)
cbar = plt.colorbar(im, ax=ax, label='Relative yield')
cbar.set_ticks([-1, -0.5, 0, 0.5, 1])

# Zero contour (equal yields)
ax.contour(angle_mesh, zeta_mesh, rel_yield.T, levels=[0.0],
           colors='black', linewidths=1.5, linestyles='-')

# Validity contours for triangle (n=3) and square (n=4)
for n, color, label in [(3, 'cyan', 'tri valid'), (4, 'lime', 'sq valid')]:
    if validity.get(n) is not None:
        ax.contour(angle_mesh, zeta_mesh, validity[n].T, levels=[0.5],
                   colors=color, linewidths=1.2, linestyles='--')

ytick_log = np.linspace(zeta_log_edges[0], zeta_log_edges[-1], 5)
ax.set_yticks(ytick_log)
ax.set_yticklabels([f'{10**v:.2g}' for v in ytick_log])

ax.set_xlabel('Opening angle (deg)')
ax.set_ylabel('ζ  (selectivity)')
ax.set_title('')

# Manual legend proxies
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='black', lw=1.5, label='Zero contour (Y_tri = Y_sq)'),
    Line2D([0], [0], color='cyan',  lw=1.2, ls='--', label='Triangle valid boundary'),
    Line2D([0], [0], color='lime',  lw=1.2, ls='--', label='Square valid boundary'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('yield_panel_B.png', dpi=150, bbox_inches='tight')
plt.show()